In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler

In [ ]:
from pathlib import Path

dataset_name = "youtube_static.csv"
data_file = Path("../data/filtered") / dataset_name

df = pd.read_csv(data_file, parse_dates=["DATE"], index_col="DATE")
print("Loaded:", data_file)

In [ ]:
df = df[['mac_dl_brate']]

target = 'mac_dl_brate'

num_lags = 4
for lag in range(1, num_lags + 1):
    df[f'{target}_lag_{lag}'] = df[target].shift(lag)

df.dropna(inplace=True)

n = len(df)
train_size = int(n * 0.8)

train = df.iloc[:train_size]
test = df.iloc[train_size:]

In [4]:
from river import preprocessing, linear_model, optim

model = (
    preprocessing.StandardScaler() |
    linear_model.LinearRegression(optimizer=optim.SGD(1e-6))
)

In [5]:
for i in range(len(train) - 1):
    x = {"mac_dl_brate": train["mac_dl_brate"].iloc[i]}
    y = train["mac_dl_brate"].iloc[i + 1]
    model.learn_one(x, y)

In [ ]:
records = []
horizon = 96

for t in range(horizon, len(df) - horizon):

    # MULTI-HORIZON PREDICTION
    for h in range(1, horizon + 1):
        idx = t + h
        row = df.iloc[idx]

        x = {col: row[col] for col in df.columns if col != 'mac_dl_brate'}
        pred = model.predict_one(x)
        actual = row['mac_dl_brate']

        records.append({
            'timestamp': df.index[idx],
            'source_t': df.index[t],
            'horizon': h,
            'actual': actual,
            'prediction': pred
        })

    # ONLINE TRAINING ON THE NEXT TRUE VALUE
    train_row = df.iloc[t + 1]
    x_train = {col: train_row[col] for col in df.columns if col != 'mac_dl_brate'}
    y_train = train_row['mac_dl_brate']

    model.learn_one(x_train, y_train)


In [ ]:
actuals = []
predictions = []

test_values = test["mac_dl_brate"].values
horizon = 96

df_rec = pd.DataFrame(records)
rmse = np.sqrt(((df_rec["prediction"] - df_rec["actual"])**2).mean())
mae  = (df_rec["prediction"] - df_rec["actual"]).abs().mean()



In [ ]:
scaler = MinMaxScaler()
scaler.fit(train["mac_dl_brate"].values.reshape(-1, 1))

actual = df_rec["actual"].values.reshape(-1, 1)
pred   = df_rec["prediction"].values.reshape(-1, 1)


actual_scaled = scaler.transform(actual)
pred_scaled   = scaler.transform(pred)

rmse_scaled = np.sqrt(mean_squared_error(actual_scaled, pred_scaled))
mae_scaled  = mean_absolute_error(actual_scaled, pred_scaled)

print("Scaled RMSE:", rmse_scaled)
print("Scaled MAE :", mae_scaled)


In [ ]:
import matplotlib.pyplot as plt

test_start_time = df.index[train_size] 

df_forecasts = pd.DataFrame(df_rec)

df_test_forecasts = df_forecasts[df_forecasts['source_t'] >= test_start_time]

df_agg = df_test_forecasts.groupby('source_t').agg({
    'actual': 'mean',
    'prediction': 'mean'
}).dropna().reset_index()

df_agg['source_t'] = pd.to_datetime(df_agg['source_t'])

plt.plot(df_agg['source_t'], df_agg['actual'], label='Actual')
plt.plot(df_agg['source_t'], df_agg['prediction'], label='Predicted')
plt.xlabel('Timestamp')
plt.ylabel('Downlink Bitrate')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
results_dir = Path("../results/metrics")
results_dir.mkdir(parents=True, exist_ok=True)

metrics_df = pd.DataFrame([{
    "model": "OLR",
    "setting": "univariate",
    "dataset": "youtube_static",
    "rmse": rmse_scaled,
    "mae": mae_scaled,
}])

metrics_file = results_dir / "olr_uni_metrics.csv"
metrics_df.to_csv(metrics_file, index=False)

print("Saved metrics to:", metrics_file)